# Netflix Analytics — código corregido y documentado

**Objetivo:** limpiar y analizar eventos de visualización, calcular métricas de finalización, anonimizar identificadores, generar gráficos y exportar reportes.

Este notebook también documenta las decisiones de código limpio, optimización, modularidad, depuración y pruebas solicitadas en la consigna.

**Importante:** no se multiplican artificialmente los minutos vistos. Los datos originales se conservan; la tasa se limita al 100 % para evitar valores superiores a la duración total del contenido.

## 1. Malas prácticas identificadas y mejoras aplicadas

| Problema que se debe evitar | Mejora aplicada |
|---|---|
| Todo el procesamiento en un único bloque | Separación en funciones y módulos (`carga_datos`, `preprocesamiento`, `metricas`, `visualizacion`) |
| Nombres poco descriptivos | Nombres claros como `calcular_tasa_finalizacion` |
| Repetición de operaciones | Funciones reutilizables para limpieza, métricas y exportación |
| Bucles innecesarios sobre filas | Operaciones vectorizadas de pandas y NumPy |
| Listas construidas manualmente | Comprensiones de listas para extraer títulos únicos |
| Lectura de archivos sin validación | Comprobación de columnas y conversión segura de tipos |
| Ausencia de documentación | Docstrings en funciones y archivo `README.md` |
| Validaciones informales solamente | Pruebas unitarias con `pytest` y aserciones del pipeline |

## 2. Optimización del rendimiento

- **Pandas/NumPy vectorizados:** cálculos de tasas y filtros sin recorrer fila por fila.
- **Diccionario:** conversión del nombre del plan a su nivel en tiempo constante promedio.
- **Comprensión de listas:** extracción clara de títulos únicos.
- **Generador:** lectura de JSONL línea por línea para no cargar primero todo el archivo como texto.
- **GroupBy:** agregación de métricas por título en una sola operación.

In [ ]:
!pip -q install pandas numpy matplotlib openpyxl pyarrow pytest pytest-cov flake8

## 3. Carga de datos

Sube el archivo de eventos en formato `.jsonl` o `.txt`. El cargador admite JSONL puro y archivos de texto que contengan una línea JSON por evento (con el campo `usuario_id`).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.colab import files

COLUMNAS_REQUERIDAS = {
    "usuario_id", "edad", "genero", "titulo", "minutos_vistos",
    "duracion_contenido_min", "plan", "dispositivo", "pais",
    "fecha_visualizacion"
}

def iterar_eventos(ruta):
    """Genera registros JSON válidos leyendo el archivo línea por línea.

    Ignora texto que no corresponda a un objeto JSON con usuario_id.
    """
    with open(ruta, encoding="utf-8") as archivo:
        for linea in archivo:
            linea = linea.strip()
            if not linea.startswith("{"):
                continue
            try:
                registro = json.loads(linea)
            except json.JSONDecodeError:
                continue
            if isinstance(registro, dict) and "usuario_id" in registro:
                yield registro

def leer_eventos(ruta):
    """Carga los eventos JSONL en un DataFrame y valida su esquema."""
    datos = pd.DataFrame(iterar_eventos(ruta))
    if datos.empty:
        raise ValueError("No se encontraron registros JSON válidos en el archivo.")

    faltantes = COLUMNAS_REQUERIDAS - set(datos.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas obligatorias: {sorted(faltantes)}")
    return datos

print("Selecciona el archivo eventos_visualizacion.jsonl o .txt")
subidos = files.upload()
nombre_archivo = next(iter(subidos))
df_original = leer_eventos(nombre_archivo)

print(f"Registros cargados: {len(df_original)}")
display(df_original.head())

## 4. Funciones de limpieza y transformación

Se convierten los campos numéricos antes de filtrar nulos. La duración igual a cero **no elimina el registro**: se conserva y su tasa de finalización queda como `NaN`, porque no es posible dividir entre cero.

In [ ]:
NIVEL_PLAN = {"Básico": 1, "Estándar": 2, "Premium": 3}

def limpiar_datos(datos):
    """Limpia tipos y filtra registros incompletos y usuarios menores de 18.

    Conserva eventos con duración cero para que la tasa de finalización
    pueda quedar indefinida (NaN), sin provocar división por cero.
    """
    df_limpio = datos.copy()

    columnas_numericas = [
        "edad", "minutos_vistos", "duracion_contenido_min"
    ]
    for columna in columnas_numericas:
        df_limpio[columna] = pd.to_numeric(
            df_limpio[columna], errors="coerce"
        )

    columnas_obligatorias = [
        "usuario_id", "edad", "genero", "titulo", "minutos_vistos",
        "duracion_contenido_min", "plan", "dispositivo", "pais",
        "fecha_visualizacion"
    ]
    df_limpio = df_limpio.dropna(subset=columnas_obligatorias)
    df_limpio = df_limpio.loc[df_limpio["edad"] >= 18].copy()

    if (df_limpio["duracion_contenido_min"] < 0).any():
        raise ValueError("Hay duraciones negativas en el conjunto de datos.")
    if (df_limpio["minutos_vistos"] < 0).any():
        raise ValueError("Hay minutos vistos negativos en el conjunto de datos.")

    return df_limpio


def agregar_nivel_plan(datos):
    """Añade nivel_plan: Básico=1, Estándar=2 y Premium=3."""
    resultado = datos.copy()
    resultado["nivel_plan"] = resultado["plan"].map(NIVEL_PLAN)
    return resultado


def anonimizar_usuarios(datos):
    """Sustituye los identificadores originales por códigos enteros locales."""
    resultado = datos.copy()
    resultado["usuario_id"] = (
        pd.factorize(resultado["usuario_id"], sort=True)[0] + 1
    )
    return resultado

## 5. Métricas y resumen

La tasa se expresa como porcentaje (0–100). `alta_finalizacion` significa una tasa igual o superior al 80 %. Este umbral es un criterio analítico elegido para el ejercicio, no un dato proporcionado por Netflix.

In [ ]:
def calcular_tasa_finalizacion(minutos_vistos, duracion):
    """Calcula la finalización porcentual; devuelve NaN si la duración es 0.

    El resultado se limita al intervalo 0–100.
    """
    tasa = (minutos_vistos / duracion.replace(0, np.nan)) * 100
    return tasa.clip(lower=0, upper=100)


def agregar_metricas(datos, umbral=80):
    """Agrega tasa_finalizacion y la bandera alta_finalizacion."""
    resultado = datos.copy()
    resultado["tasa_finalizacion"] = calcular_tasa_finalizacion(
        resultado["minutos_vistos"],
        resultado["duracion_contenido_min"]
    )
    resultado["alta_finalizacion"] = (
        resultado["tasa_finalizacion"].ge(umbral).fillna(False)
    )
    return resultado


def crear_resumen(datos):
    """Resume visualizaciones, minutos y finalización media por título."""
    return (
        datos.groupby("titulo", as_index=False)
        .agg(
            visualizaciones=("titulo", "size"),
            minutos_totales=("minutos_vistos", "sum"),
            finalizacion_media=("tasa_finalizacion", "mean")
        )
        .sort_values("minutos_totales", ascending=False)
    )

## 6. Ejecutar el pipeline

El orden de las operaciones es explícito y cada función trabaja sobre una copia para evitar modificaciones inesperadas del DataFrame original.

In [ ]:
df = limpiar_datos(df_original)
df = agregar_nivel_plan(df)
df = agregar_metricas(df, umbral=80)
df = anonimizar_usuarios(df)

# Comprensión de listas: títulos únicos sin valores nulos.
titulos_unicos = [
    titulo for titulo in df["titulo"].dropna().unique()
]

resumen = crear_resumen(df)

print("Registros originales:", len(df_original))
print("Registros después de limpieza:", len(df))
print("Títulos diferentes:", len(titulos_unicos))
print("Géneros diferentes:", df["genero"].nunique())
print("Tasa media de finalización:", round(df["tasa_finalizacion"].mean(), 2), "%")
display(df.head())
display(resumen.head(10))

## 7. Visualizaciones

In [ ]:
def graficar_resultados(resumen, datos):
    """Muestra los títulos con más minutos vistos y la distribución de planes."""
    top = resumen.head(10)

    top.plot(
        x="titulo", y="minutos_totales", kind="bar",
        figsize=(12, 5), legend=False,
        title="Top 10 contenidos por minutos vistos"
    )
    plt.xlabel("Contenido")
    plt.ylabel("Minutos vistos")
    plt.xticks(rotation=60, ha="right")
    plt.tight_layout()
    plt.show()

    datos["plan"].value_counts().plot(
        kind="pie", autopct="%1.1f%%", figsize=(6, 6),
        title="Distribución de eventos por plan"
    )
    plt.ylabel("")
    plt.show()

graficar_resultados(resumen, df)

## 8. Pruebas y validaciones

Se incluyen pruebas unitarias de limpieza y métricas, además de validaciones del resultado completo. La duración cero debe producir `NaN`, no infinito ni error.

In [ ]:
# Pruebas rápidas ejecutables dentro del notebook
def ejecutar_pruebas():
    """Ejecuta verificaciones básicas y detiene el proceso si alguna falla."""
    # Tasa normal y duración cero
    tasas = calcular_tasa_finalizacion(
        pd.Series([50.0, 10.0]),
        pd.Series([100.0, 0.0])
    )
    assert tasas.iloc[0] == 50.0
    assert pd.isna(tasas.iloc[1])

    # Filtro de adultos y eliminación de edad nula
    muestra = pd.DataFrame({
        "usuario_id": ["A", "B", "C"],
        "edad": [17, 18, None],
        "genero": ["Drama"] * 3,
        "titulo": ["X"] * 3,
        "minutos_vistos": [10, 20, 30],
        "duracion_contenido_min": [30, 30, 30],
        "plan": ["Básico"] * 3,
        "dispositivo": ["TV"] * 3,
        "pais": ["ES"] * 3,
        "fecha_visualizacion": ["2026-01-01"] * 3
    })
    limpio = limpiar_datos(muestra)
    assert len(limpio) == 1
    assert limpio["edad"].iloc[0] == 18

    # Validaciones del DataFrame real
    assert df["edad"].ge(18).all()
    assert df["nivel_plan"].notna().all()
    assert df["usuario_id"].dtype.kind in "iu"
    assert df["tasa_finalizacion"].dropna().between(0, 100).all()
    assert "alta_finalizacion" in df.columns

    print("Todas las pruebas rápidas finalizaron correctamente.")

ejecutar_pruebas()

## 9. Exportar reportes a Excel

Se genera un único libro Excel con hojas para los registros procesados y el resumen. También se guardan CSV y Parquet para facilitar otros usos.

In [ ]:
from google.colab import files

ruta_excel = "reporte_netflix.xlsx"
with pd.ExcelWriter(ruta_excel, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="Registros", index=False)
    resumen.to_excel(writer, sheet_name="Resumen", index=False)

df.to_parquet("reporte_netflix.parquet", index=False)
resumen.to_csv("resumen_netflix.csv", index=False)

print("Reportes generados correctamente.")
files.download(ruta_excel)

## 10. Estructura modular propuesta

El notebook contiene las funciones documentadas para facilitar su ejecución en Colab. Para mantener el proyecto en formato de software, se propone separar esas funciones en los siguientes módulos:

```text
netflix_analytics/
├── data/
│   ├── raw/
│   └── processed/
├── src/
│   ├── carga_datos.py
│   ├── preprocesamiento.py
│   ├── metricas.py
│   └── visualizacion.py
├── tests/
│   └── test_metricas.py
├── main.py
├── README.md
└── requirements.txt
```

En un proyecto desarrollado fuera de Colab, `main.py` coordina el flujo y `src/` contiene la lógica. `tests/` mantiene las pruebas separadas del código de producción.

## 11. Documentación: README.md

El archivo README debe explicar el objetivo, requisitos, instalación, ejecución, estructura y decisiones de análisis. La siguiente plantilla se puede guardar como `README.md`.

In [ ]:
%%writefile README.md
# Netflix Analytics

## Objetivo
Procesar eventos de visualización para analizar consumo de contenidos,
planes y tasa de finalización.

## Requisitos
Python 3.10 o superior. Dependencias: pandas, numpy, matplotlib,
openpyxl, pyarrow y pytest.

## Instalación
```bash
pip install pandas numpy matplotlib openpyxl pyarrow pytest
```

## Ejecución
Ejecutar las celdas del notebook `Colab_NETFLIX_corregido.ipynb`
en orden y subir el archivo JSONL/TXT de eventos.

## Estructura
- `data/raw/`: datos de entrada.
- `data/processed/`: resultados procesados.
- `src/`: carga, limpieza, métricas y visualización.
- `tests/`: pruebas automatizadas.
- `README.md`: documentación del proyecto.

## Reglas de procesamiento
- Se excluyen registros con campos obligatorios nulos.
- Se conservan usuarios de 18 años o más.
- Los eventos con duración cero se conservan; su tasa queda como NaN.
- La alta finalización se define como una tasa >= 80%.
- Los identificadores se reemplazan por códigos enteros locales.

## Salidas
- `reporte_netflix.xlsx`: hojas Registros y Resumen.
- `reporte_netflix.parquet`: registros procesados.
- `resumen_netflix.csv`: resumen por título.

## 12. Dependencias del proyecto

In [ ]:
%%writefile requirements.txt
pandas
numpy
matplotlib
openpyxl
pyarrow
pytest
pytest-cov
flake8

## 13. Plan de depuración y pruebas

1. **Inspección inicial:** revisar `df.shape`, `df.dtypes`, `df.isna().sum()` y `df.head()` después de cargar.
2. **Depuración interactiva:** usar puntos de interrupción (`breakpoint()`), ejecutar celda por celda y revisar variables intermedias.
3. **Pruebas unitarias:** comprobar por separado la limpieza, la conversión de planes y el cálculo de tasas.
4. **Pruebas de integración:** ejecutar carga → limpieza → métricas → resumen → exportación con un archivo de prueba.
5. **Pruebas de límites:** edad 17/18, duración 0, valores nulos y minutos vistos mayores que la duración.
6. **Pruebas de regresión:** conservar ejemplos de entrada y comparar que las métricas esperadas no cambien sin justificación.
7. **Calidad del código:** ejecutar `pytest -q`, `flake8 src tests` y revisar docstrings.

**Herramientas:** depurador de Colab/Jupyter, `breakpoint()` de Python, `pytest` para pruebas y `flake8` para estilo.

## 14. Comprobación final

In [ ]:
print("ANÁLISIS DE NETFLIX COMPLETADO")
print("Registros analizados:", len(df))
print("Contenidos diferentes:", df["titulo"].nunique())
print("Usuarios codificados:", df["usuario_id"].nunique())
print("Alta finalización:", int(df["alta_finalizacion"].sum()))
print("Tasa media:", round(df["tasa_finalizacion"].mean(), 2), "%")
print("Archivos generados: reporte_netflix.xlsx, reporte_netflix.parquet, resumen_netflix.csv")